# Model B: 1D CNN (+ hyperparameter study)

Textbook Ch 10 (convolution; 1D conv notebooks 10.1-10.2). Filters of width 3/4/5 slide over the word embeddings (n-gram detectors) followed by max-pooling. This notebook also holds the hyperparameter study (optimizer and dropout).

**Setup:** Runtime > Change runtime type > GPU (T4). Run all cells top to bottom. Results are appended to `results/results_B.csv`; do NOT edit `common.py` (frozen shared code).

In [ ]:
%pip install -q datasets transformers
import os
if not os.path.exists("common.py"):
    REPO_URL = "https://github.com/ZhongVang/csci167-rotten-tomatoes.git"   # group repo (private until submission)
    !git clone $REPO_URL project
    %cd project

In [ ]:
# Optional (Colab): keep results on Drive so they survive session timeouts.
# Must run BEFORE `import common`.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/CSCI167_results"
except Exception:
    pass  # running locally: results go to ./results

In [ ]:
OWNER = "B"
RESULTS_FILE = f"results_{OWNER}.csv"
SEEDS = [0, 1, 2]

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common import *

print(save_env_info(OWNER))                  # versions + hardware for the report
data = load_data()
print({k: len(v[0]) for k, v in data.items()}, "-> train/val/test sizes")
print("label counts (train):", np.bincount(data["train"][1]))

## Model

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, n_filters=100, kernel_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.min_len = max(kernel_sizes)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(kernel_sizes), 2)

    def forward(self, ids, mask):
        e = self.emb(ids)                                   # B, L, D
        if e.size(1) < self.min_len:                        # very short batches
            e = F.pad(e, (0, 0, 0, self.min_len - e.size(1)))
        e = e.transpose(1, 2)                               # B, D, L
        feats = [F.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(feats, dim=1)))

set_seed(0)
_, _, _, v = make_vocab_loaders(data, 64, 0)
m = TextCNN(len(v)); print(m); print("trainable / total params:", count_params(m))

## Main runs (reference config: Adam lr 1e-3, dropout 0.5), 3 seeds, plus reduced-data runs

In [ ]:
def run(seed, frac, optimizer="adam", lr=1e-3, dropout=0.5, tag="cnn"):
    set_seed(seed)
    train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, 64, seed, train_frac=frac)
    model = TextCNN(len(vocab), dropout=dropout)
    return train_and_eval(model, (train_dl, val_dl, test_dl), run_name=f"{tag}_f{frac}_s{seed}",
                          model_name="TextCNN", seed=seed, optimizer=optimizer, lr=lr,
                          epochs=20, patience=5, batch_size=64, train_frac=frac,
                          extra={"dropout": dropout}, results_file=RESULTS_FILE)

for frac in [1.0, 0.25, 0.1]:
    for seed in SEEDS:
        run(seed, frac)

## Hyperparameter study (full train set, 3 seeds each)

One factor changed at a time relative to the reference (Adam, lr 1e-3, dropout 0.5):
- **Optimizer:** Adam vs SGD+momentum (lr 0.05 is a first guess; consider checking 0.01/0.1 on validation).
- **Dropout:** 0.0 vs 0.5 (reference) 

Select settings on VALIDATION accuracy only. Note convergence speed (best epoch), stability across seeds, and time.

In [ ]:
configs = [
    dict(tag="cnn_sgd",    optimizer="sgd",  lr=0.05, dropout=0.5),   # factor 1: optimizer
    dict(tag="cnn_do0.0",  optimizer="adam", lr=1e-3, dropout=0.0),   # factor 2: dropout
]
for cfg in configs:
    for seed in SEEDS:
        run(seed, 1.0, **cfg)

In [ ]:
summarize(f'results_{OWNER}.csv')

## Results: summary table, learning curves, error analysis

In [ ]:
summarize(f'results_{OWNER}.csv')   # mean +/- std over seeds, per configuration

In [ ]:
plot_history("cnn_f1.0_s0", "TextCNN: train vs validation (seed 0)");

In [ ]:
import json
h = json.load(open(f"{RESULTS_DIR}/history/cnn_f1.0_s0.json"))
preds, labels = np.array(h["test_preds"]), np.array(data["test"][1])
texts = data["test"][0]
cm = np.zeros((2, 2), int)
for y, p in zip(labels, preds): cm[y, p] += 1
print("confusion matrix (rows=true neg/pos, cols=pred neg/pos):\n", cm)
wrong = np.where(preds != labels)[0]
print(len(wrong), "errors. Examples (true, pred, text):")
for i in wrong[:20]: print(labels[i], preds[i], "|", texts[i])
# TODO for the report: what do the errors have in common? (sarcasm, negation, mixed reviews, very short text...)